# Imports

In [53]:
import pickle
from random import shuffle
from PIL import Image
import torch
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
from sklearn.model_selection import train_test_split
import numpy as np

# Using Pickle Files And Testing Whether Files Are Correct

In [44]:
negative_pairs_path = "C:/Active-Repositories/secure-signature-verification/image_pairs/negative_pairs.pkl"
positive_pairs_path = "C:/Active-Repositories/secure-signature-verification/image_pairs/positive_pairs.pkl"
with open(f'{negative_pairs_path}','rb') as f1:
    negative_pairs = pickle.load(f1)
with open(f'{positive_pairs_path}','rb') as f2:
    positive_pairs = pickle.load(f2)
lnp = len(negative_pairs)
lpp = len(positive_pairs)

In [30]:
print(lnp+lpp)

38400


# Combining Pairs Into One List And Shuffling For Training

In [45]:
all_pairs = negative_pairs + positive_pairs

In [46]:
shuffle(all_pairs)

# Model

In [47]:
train_pairs, test_pairs = train_test_split(all_pairs, test_size=0.2, random_state=42)

In [52]:
class SiameseSignatureDataset(Dataset):
    def __init__(self, pairs):
        self.pairs = pairs
    def __len__(self):
        return len(self.pairs)
    def __getitem__(self, idx):
        img1, img2, label = self.pairs[idx]
        if isinstance(img1, Image.Image):
            img1 = np.array(img1)
        if isinstance(img2, Image.Image):
            img2 = np.array(img2)
        if img1.max() > 1.0: img1 = img1 / 255.0
        if img2.max() > 1.0: img2 = img2 / 255.0
        return (
            torch.tensor(img1, dtype=torch.float32).unsqueeze(0),  # [1, H, W]
            torch.tensor(img2, dtype=torch.float32).unsqueeze(0),
            torch.tensor(label, dtype=torch.float32)
        )

In [49]:
class SiameseNetwork(nn.Module):
    def __init__(self):
        super(SiameseNetwork, self).__init__()
        self.cnn = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=5), 
            nn.ReLU(),
            nn.MaxPool2d(2),                  
            nn.Conv2d(32, 64, kernel_size=5), 
            nn.ReLU(),
            nn.MaxPool2d(2)                   
        )
        self._to_linear = self._get_flattened_size()

        self.fc = nn.Sequential(
            nn.Linear(self._to_linear, 512),
            nn.ReLU(),
            nn.Linear(512, 128)
        )
    def _get_flattened_size(self):
        with torch.no_grad():
            dummy = torch.zeros(1, 1, 256, 256)
            out = self.cnn(dummy)
            return out.view(1, -1).shape[1]
    def forward_once(self, x):
        x = self.cnn(x)
        x = x.view(x.size(0), -1)
        x = self.fc(x)
        return x
    def forward(self, x1, x2):
        out1 = self.forward_once(x1)
        out2 = self.forward_once(x2)
        return out1, out2


In [50]:
class ContrastiveLoss(nn.Module):
    def __init__(self, margin=1.0):
        super(ContrastiveLoss, self).__init__()
        self.margin = margin
    def forward(self, out1, out2, label):
        euclidean_distance = nn.functional.pairwise_distance(out1, out2)
        loss = torch.mean(
            (1 - label) * torch.pow(euclidean_distance, 2) +
            label * torch.pow(torch.clamp(self.margin - euclidean_distance, min=0.0), 2)
        )
        return loss

In [51]:
train_dataset = SiameseSignatureDataset(train_pairs)
test_dataset = SiameseSignatureDataset(test_pairs)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = SiameseNetwork().to(device)
criterion = ContrastiveLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)


In [ ]:
num_epochs = 10
for epoch in range(num_epochs):
    model.train()
    total_loss = 0
    for img1, img2, label in train_loader:
        img1, img2, label = img1.to(device), img2.to(device), label.to(device)
        optimizer.zero_grad()
        out1, out2 = model(img1, img2)
        loss = criterion(out1, out2, label)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    avg_loss = total_loss / len(train_loader)
    print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {avg_loss:.4f}")


Epoch [1/10], Loss: 0.2467
Epoch [2/10], Loss: 0.2225
Epoch [3/10], Loss: 0.2028
Epoch [4/10], Loss: 0.1948
Epoch [5/10], Loss: 0.1895
Epoch [6/10], Loss: 0.1869
Epoch [7/10], Loss: 0.1841
Epoch [8/10], Loss: 0.1811
Epoch [9/10], Loss: 0.1786
Epoch [10/10], Loss: 0.1779


In [ ]:
model.eval()
correct = 0
total = 0
threshold = 0.5
with torch.no_grad():
    for img1, img2, label in test_loader:
        img1, img2, label = img1.to(device), img2.to(device), label.to(device)
        out1, out2 = model(img1, img2)
        distance = nn.functional.pairwise_distance(out1, out2)
        predictions = (distance < threshold).float()
        correct += (predictions == label).sum().item()
        total += label.size(0)
print(f"Test Accuracy: {100 * correct / total:.2f}%")